In [1]:
import json, os
from pathlib import Path
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


PROJECT = Path(r"C:\Users\shlok\projects\ddp-llm")
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = PROJECT / "checkpoints" / "qwen1.5b-qlora-v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

torch: 2.11.0+cu128 cuda: True


In [2]:
SYSTEM_PROMPT = """You are a parser that converts a user's natural language response into a subset of the shown options.

The user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.

Output format (JSON only, nothing else):
- A JSON list of the favored labels, e.g. ["A", "B"]
- [] if the user explicitly rejects ALL options ("none of these", "all wrong")
- "*" if the utterance is off-topic OR expresses no usable preference ("I don't know", "they all look the same", "I love football")

Rules:
- Any positive signal about an option means it goes in the list.
- "X is better than Y" endorses only X, not Y.
- "X and Y are both good, X is better" endorses both X and Y.
- Negations like "not D" or "anything but B" mean the remaining options go in the list.
- Questions like "is it A?" are treated as tentative endorsement of A.

Output ONLY the JSON. No explanation, no prose."""

def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

train_raw = load_jsonl(PROJECT / "data" / "train.jsonl")
test_raw  = load_jsonl(PROJECT / "data" / "test.jsonl")

def to_chat(ex):
    user = f'Options: {ex["options"]}\nUser: {ex["utterance"]}'
    assistant = json.dumps(ex["label"])
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user},
            {"role": "assistant", "content": assistant},
        ]
    }

train_ds = Dataset.from_list([to_chat(e) for e in train_raw])
test_ds  = Dataset.from_list([to_chat(e) for e in test_raw])
print("train:", len(train_ds), "test:", len(test_ds))
print("sample:", train_ds[0])

train: 1162 test: 191
sample: {'messages': [{'role': 'system', 'content': 'You are a parser that converts a user\'s natural language response into a subset of the shown options.\n\nThe user is shown 4 options labeled A, B, C, D and gives feedback about which are close to what they want. Your job is to output which options the user views favorably.\n\nOutput format (JSON only, nothing else):\n- A JSON list of the favored labels, e.g. ["A", "B"]\n- [] if the user explicitly rejects ALL options ("none of these", "all wrong")\n- "*" if the utterance is off-topic OR expresses no usable preference ("I don\'t know", "they all look the same", "I love football")\n\nRules:\n- Any positive signal about an option means it goes in the list.\n- "X is better than Y" endorses only X, not Y.\n- "X and Y are both good, X is better" endorses both X and Y.\n- Negations like "not D" or "anything but B" mean the remaining options go in the list.\n- Questions like "is it A?" are treated as tentative endorsem

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
model = prepare_model_for_kbit_training(model)
print("model loaded. VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

W0701 18:49:59.406000 7088 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


model loaded. VRAM (GB): 1.620461568


In [4]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [5]:
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=512,
    packing=False,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/1162 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1162 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/191 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/191 [00:00<?, ? examples/s]

In [6]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.067186,0.062623,0.055660,326172.000000,0.986882
2,0.056433,0.057797,0.056982,652344.000000,0.987710
3,0.052227,0.057525,0.051672,978516.000000,0.987636


TrainOutput(global_step=219, training_loss=0.17366257133004873, metrics={'train_runtime': 2630.1848, 'train_samples_per_second': 1.325, 'train_steps_per_second': 0.083, 'total_flos': 7886502428332032.0, 'train_loss': 0.17366257133004873, 'epoch': 3.0})

In [6]:
print("bf16:", trainer.args.bf16)
print("fp16:", trainer.args.fp16)
print("half precision backend:", trainer.args.half_precision_backend)

bf16: True
fp16: False


AttributeError: 'SFTConfig' object has no attribute 'half_precision_backend'

In [7]:
for n, p in model.named_parameters():
    if p.requires_grad:
        print(n, p.dtype)
        break

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.bfloat16


In [7]:
from peft import PeftModel

# Base model in 4-bit
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)

# Load adapter
adapter_path = str(OUTPUT_DIR)  # or a specific checkpoint subfolder
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)

      README.md  (0.0 MB)
[dir] checkpoint-146/  (11 files)
[dir] checkpoint-219/  (11 files)
[dir] checkpoint-73/  (11 files)


In [8]:
import gc, torch
try:
    del trainer, model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("VRAM after cleanup:", torch.cuda.memory_allocated() / 1e9, "GB")

VRAM after cleanup: 0.953521152 GB


In [9]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
)
adapter_path = str(OUTPUT_DIR / "checkpoint-219")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
print("adapter loaded from", adapter_path)
print("VRAM (GB):", torch.cuda.memory_allocated() / 1e9)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


adapter loaded from C:\Users\shlok\projects\ddp-llm\checkpoints\qwen1.5b-qlora-v1\checkpoint-219
VRAM (GB): 2.18086912


In [10]:
def generate_ft(messages, max_new_tokens=40):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

test_cases = [
    ("A and C look good", '["A", "C"]'),
    ("not D", '["A", "B", "C"]'),
    ("I don't know", '"*"'),
    ("none of these work", '[]'),
    ("A is better than B", '["A"]'),
    ("what's for lunch", '"*"'),
    ("hate all of them", '[]'),
    ("A great, B bad, C great, D bad", '["A", "C"]'),
]

for utt, expected in test_cases:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: ["A", "B", "C", "D"]\nUser: {utt}'},
    ]
    got = generate_ft(msgs)
    match = "✓" if got.strip() == expected else "✗"
    print(f"{match} {utt!r}\n   expected: {expected}\n   got:      {got}\n")

C:\Users\shlok\projects\ddp-llm\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✓ 'A and C look good'
   expected: ["A", "C"]
   got:      ["A", "C"]

✓ 'not D'
   expected: ["A", "B", "C"]
   got:      ["A", "B", "C"]

✓ "I don't know"
   expected: "*"
   got:      "*"

✓ 'none of these work'
   expected: []
   got:      []

✓ 'A is better than B'
   expected: ["A"]
   got:      ["A"]

✓ "what's for lunch"
   expected: "*"
   got:      "*"

✓ 'hate all of them'
   expected: []
   got:      []

✓ 'A great, B bad, C great, D bad'
   expected: ["A", "C"]
   got:      ["A", "C"]



In [11]:
from tqdm import tqdm

def parse_output(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").lstrip("json").strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return None
    if parsed == "*" or (isinstance(parsed, list) and all(x in ["A","B","C","D"] for x in parsed)):
        return parsed
    return None

def labels_equal(a, b):
    if a == "*" or b == "*":
        return a == b
    if isinstance(a, list) and isinstance(b, list):
        return set(a) == set(b)
    return False

test_examples = load_jsonl(PROJECT / "data" / "test.jsonl")

results = []
for ex in tqdm(test_examples):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Options: {ex["options"]}\nUser: {ex["utterance"]}'},
    ]
    raw = generate_ft(msgs)
    parsed = parse_output(raw)
    correct = parsed is not None and labels_equal(parsed, ex["label"])
    results.append({
        "utterance": ex["utterance"],
        "category": ex.get("category", "unknown"),
        "gold": ex["label"],
        "raw": raw,
        "parsed": parsed,
        "correct": correct,
        "valid_format": parsed is not None,
    })

n = len(results)
n_valid = sum(r["valid_format"] for r in results)
n_correct = sum(r["correct"] for r in results)
print(f"\nformat validity: {n_valid}/{n} = {n_valid/n:.1%}")
print(f"exact match:     {n_correct}/{n} = {n_correct/n:.1%}")

from collections import defaultdict
by_cat = defaultdict(lambda: [0, 0])
for r in results:
    by_cat[r["category"]][0] += 1
    by_cat[r["category"]][1] += int(r["correct"])
print("\nper category:")
for cat, (total, correct) in sorted(by_cat.items()):
    print(f"  {cat}: {correct}/{total} = {correct/total:.1%}")

100%|██████████| 191/191 [01:49<00:00,  1.75it/s]


format validity: 191/191 = 100.0%
exact match:     181/191 = 94.8%

per category:
  comparative: 25/26 = 96.2%
  mixed_sentiment: 27/28 = 96.4%
  multi_positive: 12/13 = 92.3%
  negation: 36/42 = 85.7%
  off_topic: 22/22 = 100.0%
  reject_all: 29/29 = 100.0%
  single_positive: 9/9 = 100.0%
  uncertainty: 21/22 = 95.5%


In [12]:
print("=== FAILURES ===")
for r in results:
    if not r["correct"]:
        print(f"[{r['category']}] {r['utterance']!r}")
        print(f"   gold:   {r['gold']}")
        print(f"   parsed: {r['parsed']}")
        print(f"   raw:    {r['raw']!r}")
        print()

=== FAILURES ===
[negation] 'except for C, everything is a miss'
   gold:   ['C']
   parsed: ['A', 'B', 'D']
   raw:    '["A", "B", "D"]'

[multi_positive] 'A is okay but B is closer'
   gold:   ['A', 'B']
   parsed: ['B']
   raw:    '["B"]'

[mixed_sentiment] 'A: 1. B: 1. C: 9. D: 8.'
   gold:   ['C', 'D']
   parsed: ['A']
   raw:    '["A"]'

[negation] 'except B, all are off'
   gold:   ['B']
   parsed: ['A', 'C', 'D']
   raw:    '["A", "C", "D"]'

[negation] 'none of A'
   gold:   ['B', 'C', 'D']
   parsed: []
   raw:    '[]'

[uncertainty] 'cant commit to any of these'
   gold:   *
   parsed: []
   raw:    '[]'

[negation] 'not the second one'
   gold:   ['A', 'C', 'D']
   parsed: ['A', 'C']
   raw:    '["A", "C"]'

[comparative] 'B leaves D in the dust'
   gold:   ['B']
   parsed: ['A', 'C']
   raw:    '["A", "C"]'

[negation] 'second one is wrong'
   gold:   ['A', 'C', 'D']
   parsed: ['A', 'B', 'C']
   raw:    '["A", "B", "C"]'

[negation] 'A can go'
   gold:   ['B', 'C', 'D']
 